In [ ]:
import rmllm
import json
import pandas as pd

In [ ]:
data_dir = rmllm.config.EXTERNAL_DATA_DIR/"exp2"
models = ["gemma3_12b","gemma3_12b-it-qat","gemma3_27b","gemma3_27b-it-qat","llama3.3_70b","llama4_scout"]
tasks = ["5t_100","5t_fb_100","10t_100","10t_fb_100"]

In [ ]:
task_map = {
    "5t_100": "task_1",
    "10t_100": "task_2",
    "5t_fb_100":"task_3",
    "10t_fb_100":"task_4"
}

exp_map = {
    "task_1":1,
    "task_2":2, 
    "task_3":3,
    "task_4":4
}

setsize_map = {
    "task_1":20,
    "task_2":40, 
    "task_3":20,
    "task_4":40, 
}
fb_map ={
    "task_1":False,
    "task_2":False, 
    "task_3":True,
    "task_4":True,
}


In [ ]:
def make_tabular(data):
    x = data["taskdata"]
    y = []
    for i in x:
        y.append(
            {
                "tridx": i["trial_idx"],
                "trcode": i["trcode"],
                "source": i["context_item"],
                "trace": i["trace_id"],
                **(eval(i["pred_resp"]["content"])),
                **(i["stimulus"]),
                "fb_inputs": json.loads(i["inputs"][0]["content"]).get("Feedback on previous response","")
                }
        )
    return y


In [ ]:
y = {}
z = pd.DataFrame()
for model in models:
    y[model] = {}
    for task in tasks:
        task_dir = data_dir/task/f"ollama_{model}_Convo"
        y[model][task] = None 
        if task_dir.is_dir():
            print(task_dir)
            for file in task_dir.iterdir():
                if "-tunnel" not in file.name:
                    with open(file) as f:
                        data = json.load(f)
                        print(f)
                    y[model][task] = make_tabular(data)
                    y[model][task] = pd.DataFrame(y[model][task])
                    y[model][task]["model"] = model
                    y[model][task]["task"] = task

                    z = pd.concat([z, y[model][task]])

In [ ]:
z["expid"] = z["task"].map(task_map)
z["setsize"] = z["expid"].map(setsize_map)
z["fb_exp"] = z["expid"].map(fb_map)
import numpy as np
z.loc[:, "subtask"] = z["Test_Word"].apply(
    lambda x: "phase2" if pd.notna(x) else "phase1"
)

z1 = z.loc[z["subtask"]=="phase1"]
z2 = z.loc[z["subtask"]=="phase2"]

exp_corr = {
    "test:imagined": "internal",
    "test:perceived": "external"
}

z2["corrAns"] = z2["source"].map(exp_corr)
z2["accuracy"] = z2["Judgment"] == z2["corrAns"]
z2["word1"] = z2["Test_Word"]


exp_groups = ["setsize","fb_exp","model","trace","source"]

In [ ]:
z1["word1"] = z1["Word_Pair"].apply(lambda x: x["word_1"])
z1["word2"] = z1["Word_Pair"].apply(lambda x: x["word_2"])

z1_perceived = z1.loc[z1["source"]=="perceived"]
z1_imagined = z1.loc[z1["source"] == "imagined"]

z1_perceived["word2acc"] = z1_perceived["Word_2"] == z1_perceived["word2"]
z1_imagined["word2acc"] = z1_imagined["Word_2"] != z1_imagined["word1"]

sim_in =[ "setsize","fb_exp","model","trace"]
sim_grp = z2.groupby(exp_groups)



In [ ]:
z1_updated = pd.concat([z1_perceived, z1_imagined])

z_p1_p2_horizontal_merge = pd.merge(
    z2,
    z1_updated,
    
    how="left",
    on=["setsize","fb_exp","model","trace","word1"], 
    suffixes=('_test', '_ort'),
    validate="one_to_one"
)

In [ ]:
pd.set_option('display.max_columns', None)
z_p1_p2_horizontal_merge.groupby(["setsize","fb_exp","model","trace"]).size()

In [ ]:
# get the data with only relevant columns
exp_data = z_p1_p2_horizontal_merge.dropna(axis=1, how='all')

In [ ]:
expanded_feedback = exp_data["fb_inputs_ort"].apply(pd.Series).drop(columns=0)
shifted = [i+"_aligned" for i in expanded_feedback.columns]
expanded_feedback[shifted]= expanded_feedback[expanded_feedback.columns].shift(-1)
exp_data = pd.concat([exp_data, expanded_feedback], axis=1)

In [ ]:
def set_trialfb_quality(x):
    if isinstance(x,str):
        if "INCORRECT" in x:
            return False
        elif "None" in x:
            return False
        elif "NaN" in x:
            return False
        else:
            return True
    else:
        return None


In [ ]:
exp_data["fb_trial_quality"] = exp_data["overall feedback_aligned"].apply(lambda x: set_trialfb_quality(x))

In [ ]:
exp_data.insert(0,"order",1)

In [ ]:
exp_data.to_csv(rmllm.config.PROCESSED_DATA_DIR/"rmai_exp100_2.csv",index=False)

In [ ]:
exp_data.groupby(["model", "setsize", "fb_exp"])["trace"].nunique()